# R7 — Qwen2.5-7B-Instruct 이질적 반복

R5의 검증된 셀 0~8을 상속하고, `BASE`만 `unsloth/Qwen2.5-7B-Instruct`로 바꾼다.
arm0·arm1·arm2·arm3을 seed 42에서 실행하며, 학습 arm은 세 라운드 뒤 r3만 네 렌즈로 평가한다.

이 실행이 성공하기 전에는 원고에서 **“한 레시피를 넘어 반복되었다”**고 말할 수 없다.
성공하더라도 두 모델 계열·단일 seed의 이질적 반복이므로 모델 및 seed 일반화를 입증하지 않는다.

★**이 파일을 직접 수정하지 말 것** — `src/make_nb_r7.py`가 생성한다.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Qwen2.5-7B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None, max_new_tokens=256, out_suffix=""):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로).
       max_new_tokens, out_suffix — ★R5 신설. 512-token 평가는 기존 256-token
                      결과와 파일명이 절대 겹치지 않게 접미사를 함께 쓴다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}")

        # ★R5: 라운드 r 머지본이 생긴 순간 r-1은 죽은 무게다. 원래는 eval/probe/mmlu가
        #   끝난 뒤에야 지웠는데, 4렌즈 eval은 약 2시간이라 그동안 32GB를 붙들고 있었다.
        #   R5 배치 2가 실제로 여기서 디스크를 터뜨렸다(Docker daemon 오류는 그 증상).
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제(학습 직후): {prev_dir}")
            prev_dir = None

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}{eval_args}")
                check(f"results/eval_{arm}_r{f}_s{seed}{out_suffix}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
            check(f"results/eval_{arm}_r{r}_s{seed}{out_suffix}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크 해제는 학습 직후로 옮겼다(R5). 여기서는 다음 라운드 기준만 갱신한다.
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True, max_new_tokens=256, out_suffix=""):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다.
       max_new_tokens, out_suffix — ★R5 신설. 생성 상한과 충돌 없는 파일 접미사."""
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
    check(f"results/eval_arm0_r0_s{seed}{out_suffix}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. ★이번 세션 — `BATCH` 하나만 바꾼다

측정값은 학습 round당 약 800초, 렌즈당 약 1,800초(`n=313`)다. 표의 시간은 순수 GPU
작업 추정치이며 설치·Qwen 모델 다운로드·압축 여유는 별도다. 모든 평가는 256 tokens,
`none,agreeable,principled,agreeable_para1` 네 렌즈이고 새 출력은 `_qwen25_7b_env76`로 분리한다.

| BATCH | 세션 | 계산 | 순수 GPU 추정 |
|---|---|---|---:|
| **1** | arm0·arm1 × seed 42 | arm0: 4렌즈; arm1: 3라운드 + 4렌즈 | 4h40m |
| **2** | arm2·arm3 × seed 42 | 각 arm: 3라운드 + 4렌즈 | 5h20m |

R7의 순수 GPU 합계는 **10시간**, 두 Kaggle 세션의 설치·다운로드·압축 여유를 더한 계획값은
**약 11시간**이다. 이미 계획된 R5의 순수 GPU 36시간에 R7이 10시간을 추가하므로 둘만 해도
**46시간**이며, 주간 약 30시간 한도 안에 함께 끝난다고 가정하지 않는다.

Qwen 경로도 `common.py`의 tokenizer chat template, system role, left padding, 그리고 scalar/복수
`eos_token_id` 처리를 그대로 사용한다. 모델별 특례를 새로 넣지 않고 `BASE`만 바꾼다.

보너스 조작 확인도 추가 비용 없이 따라온다. 최근 발견한 거절 도구의 typographic apostrophe 결함은
Llama-3.1 출력에서 관찰되었다. Qwen도 같은 문자를 내는지는 이 실행에서 직접 볼 수 있고, 각 결과
JSON에 저장되는 패턴별 fire count로 별도 평가 없이 확인할 수 있다.

In [ ]:
import json, os

# ── 리비전 게이트 ────────────────────────────────────────────────────────────
NB_NAME = "nb_r7"
NB_REV = "R7b"
_ds_revisions = json.load(open("configs/arms.json", encoding="utf-8")).get("nb_revisions", {})
_ds_rev = _ds_revisions.get(NB_NAME)
assert _ds_rev == NB_REV, (
    f"리비전 불일치: 노트북={NB_NAME}/{NB_REV} Dataset={_ds_rev}. "
    f"둘 중 하나가 낡았다 — Dataset을 다시 올리거나 노트북을 다시 Import하라")
print(f"[gate] 리비전 {NB_NAME}/{NB_REV} 일치")

UNSLOTH_PIN = "2026.7.6"
OUT_SUFFIX = "_qwen25_7b_env76"
LENSES = "none,agreeable,principled,agreeable_para1"
MAX_NEW_TOKENS = 256

# evaluator가 결과에 쓰는 것과 같은 stamp로 실행 환경을 다시 확인한다.
from common import env_versions
RUNTIME_ENV = env_versions()
assert RUNTIME_ENV.get("unsloth") == UNSLOTH_PIN, (
    f"Unsloth 실행 환경 불일치: 기대={UNSLOTH_PIN} stamp={RUNTIME_ENV}")
print(f"[gate] 실행 환경 stamp 일치: {RUNTIME_ENV}")

# ★★ 커밋마다 이 값 하나만 바꾼다. ★★
BATCH = 1

BATCHES = {
    1: [("Q01", "arm0", 42), ("Q02", "arm1", 42)],
    2: [("Q03", "arm2", 42), ("Q04", "arm3", 42)],
}
assert BATCH in BATCHES, f"BATCH는 {sorted(BATCHES)} 중 하나여야 한다"
PLAN = BATCHES[BATCH]
print(f"R7 BATCH {BATCH}: " + ", ".join(f"{n}({a} x {s})" for n, a, s in PLAN))


def snap(tag):
    """세션마다 누적 results zip을 떠서 뒤 세션이 실패해도 앞 결과를 회수한다."""
    import shutil
    zp = shutil.make_archive(f"/kaggle/working/r7_results_{tag}", "zip",
                             "/kaggle/working/experiment/results")
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)


for name, arm, seed in PLAN:
    print(f"\n===== {name}: {arm} x seed {seed} =====", flush=True)
    if arm == "arm0":
        run_baseline(seed, lenses=LENSES, mmlu=False,
                     max_new_tokens=MAX_NEW_TOKENS, out_suffix=OUT_SUFFIX)
    else:
        # 세 라운드를 순서대로 학습하되 비용이 큰 평가는 r3에서만 한다.
        run_session(arm, seed, lenses=LENSES, eval_rounds=(3,),
                    probe=False, mmlu=False, max_new_tokens=MAX_NEW_TOKENS,
                    out_suffix=OUT_SUFFIX)
    snap(f"batch{BATCH}_{name.lower()}_{arm}_s{seed}")


## 5. ★결과 회수

In [ ]:
import shutil, os, glob

# TAG는 BATCH에서 유도한다 — 노트북 내용과 zip 이름이 어긋나지 않게 한다.
TAG = f"batch{BATCH}"
zp = shutil.make_archive(f"/kaggle/working/r7_results_{TAG}", "zip",
                         "/kaggle/working/experiment/results")
print("zip:", zp, "| size MB:", round(os.path.getsize(zp)/1e6, 2))
for p in sorted(glob.glob("/kaggle/working/experiment/results/*.json")):
    print(f"  {os.path.getsize(p)/1024:7.0f} KB  {os.path.basename(p)}")


## 6. 세션이 성공했는지

BATCH별 기대 파일, 네 렌즈, 생성 메타데이터, Unsloth 환경 stamp를 모두 검사한다.
`snap()`은 각 세션 직후 실행되므로 여기서 실패해도 앞 세션 결과는 회수할 수 있다.

In [ ]:
import json, os

N = 313
UNSLOTH_PIN = "2026.7.6"
OUT_SUFFIX = "_qwen25_7b_env76"
EXPECTED_LENSES = ["none", "agreeable", "principled", "agreeable_para1"]
EXPECTED_MAX_TOKENS = 256

assert OUT_SUFFIX.startswith("_qwen25_7b_"), OUT_SUFFIX
expected = []
for _, arm, seed in PLAN:
    rtag = 0 if arm == "arm0" else 3
    expected.append(f"results/eval_{arm}_r{rtag}_s{seed}{OUT_SUFFIX}.json")

assert len(expected) == len(set(expected)), f"기대 출력 이름이 충돌한다: {expected}"
assert all(path.endswith(f"{OUT_SUFFIX}.json") for path in expected), expected

for path in expected:
    assert os.path.exists(path), f"결과가 없다: {path}"
    ev = json.load(open(path, encoding="utf-8"))
    assert ev.get("schema") == "r2", f"{path}: 스키마가 다르다"
    assert ev.get("max_new_tokens") == EXPECTED_MAX_TOKENS, (
        f"{path}: max_new_tokens={ev.get('max_new_tokens')}")
    assert ev.get("out_suffix") == OUT_SUFFIX, (
        f"{path}: out_suffix={ev.get('out_suffix')} 기대={OUT_SUFFIX}")
    assert ev.get("env", {}).get("unsloth") == UNSLOTH_PIN, (
        f"{path}: env stamp unsloth={ev.get('env', {}).get('unsloth')}; 핀={UNSLOTH_PIN}")
    assert ev.get("lenses") == EXPECTED_LENSES, f"{path}: 렌즈={ev.get('lenses')}"
    assert list(ev.get("conditions", {})) == EXPECTED_LENSES, (
        f"{path}: condition 순서/목록 불일치")

    for lens in EXPECTED_LENSES:
        condition = ev["conditions"][lens]
        records = condition["records"]
        assert condition["n"] == N and len(records) == N, (
            f"{path}/{lens}: n={condition.get('n')} records={len(records)}")
        assert all({"hit_cap", "n_new_tokens", "finish_reason"} <= set(record)
                   for record in records), f"{path}/{lens}: 생성 메타데이터가 없다"
        assert all(0 < record["n_new_tokens"] <= EXPECTED_MAX_TOKENS
                   for record in records), f"{path}/{lens}: 생성 토큰 수 범위 오류"
        n_hit_cap = sum(bool(record["hit_cap"]) for record in records)
        assert condition.get("n_hit_cap") == n_hit_cap, (
            f"{path}/{lens}: cap 집계 불일치")
        assert all(record["finish_reason"] in
                   {"eos_token", "max_new_tokens", "stopping_criteria"}
                   for record in records), f"{path}/{lens}: 알 수 없는 종료 이유"
        print(f"  {os.path.basename(path)}/{lens} cap={n_hit_cap}/{N}")

print(f"\n[gate] R7 BATCH {BATCH} 통과 — {len(expected)}개 eval 파일")
